# L80 · RAG 流水线 — 文档切片 → embedding → FAISS 检索 → LLM 生成 + 引用

**目标**：用 `sentence-transformers` 编码文档，用 FAISS 构建可语义搜索的知识库，实现 `build_rag_index` 和 `retrieve` 两个核心函数。

🔗 **Aurora 连接**：这两个函数是 `aurora.rag` 的核心；Podcast Agent 调用它们从音频笔记库检索与用户问题语义最接近的上下文片段，再交给 LLM 生成回答。

RAG（Retrieval-Augmented Generation）把「检索」和「生成」解耦：先用向量相似度从知识库里挑最相关的文档块，再把这些块塞进 prompt 让 LLM 回答。LLM 不需要记住所有知识，只需要会「看文档回答问题」。关键洞察：**Embedding 把语义距离变成了向量内积**，FAISS 让这个搜索在百万量级也极快。

In [ ]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

## 1. RAG 两阶段：Retrieval + Generation

RAG 流程分两步：
- **Retrieval**：把用户 query 编码为向量，从向量数据库里找 top-k 最相关块
- **Generation**：把检索结果拼进 prompt，让 LLM 基于「外挂记忆」生成回答

检索给 LLM 提供了动态记忆——不用微调模型，知识库随时可更新。公式上，检索目标是：
```
top-k = argsort( q · d_i )  for d_i in index
```
其中 `q` 是 query 向量，`d_i` 是第 i 个文档向量，内积越大越相关（向量归一化后等价余弦相似度）。

In [ ]:
# 演示：RAG 流程示意（伪代码，无需运行）
rag_pipeline = """
用户问题
  └─► Embedding(query)  →  向量 q
         └─► FAISS.search(q, top_k)  →  最相关的 k 段文字
                └─► LLM(prompt + 检索结果)  →  回答
"""
print(rag_pipeline)

## 2. FAISS IndexFlatIP：精确内积搜索

`faiss.IndexFlatIP(dim)` 做的是精确暴力内积搜索（IP = Inner Product）：
```
scores[i] = sum( q[j] * d_i[j]  for j in range(dim) )
```
向量先做 L2 归一化（`||v|| = 1`），内积就等于余弦相似度。

**优点**：结果精确，实现简单，百万向量毫秒级。  
**限制**：不支持删除单条向量（只能重建索引）；百亿规模需改用 `IndexIVFFlat` 等近似方法。

In [ ]:
# 演示：构建一个小型 IndexFlatIP
dim = 4
index_demo = faiss.IndexFlatIP(dim)

# 3 个随机向量，归一化
vecs = np.random.randn(3, dim).astype(np.float32)
faiss.normalize_L2(vecs)
index_demo.add(vecs)

# 用第0个向量作 query，应排第1
q = vecs[[0]].copy()
D, I = index_demo.search(q, 3)
print('distances:', D)
print('indices:  ', I)  # 期望第一个是 0（与自身相似度=1.0）

## 3. Chunking：长文档切块

大模型 context 有限，直接塞整篇文章往往超长。Chunking 把长文档切成固定大小的块：
```
chunk_size    = 512   # token 数
chunk_overlap = 50    # 相邻块重叶 token，防止上下文在边界断裂
```
举例：1000 token 的文章 → `[0:512]`, `[462:974]`, `[924:1000]` 三块（重叶50）。重叶保证同一句话不会因为恰好落在边界而被截断为两半。

In [ ]:
def naive_chunk(text: str, chunk_size: int = 512, overlap: int = 50) -> list[str]:
    """按词数粗切（演示用，生产环境用 tokenizer）"""
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunks.append(' '.join(words[start:end]))
        if end == len(words):
            break
        start += chunk_size - overlap
    return chunks

# 演示
sample = ' '.join([f'word{i}' for i in range(1200)])
chunks_demo = naive_chunk(sample, chunk_size=512, overlap=50)
print(f'总词数: 1200  →  切出 {len(chunks_demo)} 块')
print(f'焴1块: {chunks_demo[0][:40]}...')
print(f'焴2块: {chunks_demo[1][:40]}...')

## 4. ✏️ 实现 `build_rag_index(texts)`

**签名**：`texts: list[str] -> (faiss.Index, list[str])`

**推理路线**：
1. 加载 `sentence-transformers` 模型（`all-mpnet-base-v2`，输出 768 维向量）
2. `model.encode(texts)` 批量编码，得到形如 `(N, 768)` 的 float32 数组
3. `faiss.normalize_L2(vectors)` 就地 L2 归一化
4. 构建 `faiss.IndexFlatIP(768)`，调用 `index.add(vectors)`
5. 返回 `(index, texts)`（chunks 原样返回，用于后续展示）

**参考输入输出**：
```
texts = ["Chord progressions define harmonic movement.",
         "MFCCs capture timbral features of audio.",
         ...  # 100 段播客笔记
        ]
→  index.ntotal == 100
→  chunks 长度 == 100
```

<details><summary>点击查看参考实现</summary>

```python
def build_rag_index(texts: list[str]):
    model = SentenceTransformer('all-mpnet-base-v2')
    vectors = model.encode(texts, convert_to_numpy=True,
                           normalize_embeddings=False).astype(np.float32)
    faiss.normalize_L2(vectors)
    index = faiss.IndexFlatIP(vectors.shape[1])
    index.add(vectors)
    return index, texts
```

</details>

In [ ]:
def build_rag_index(texts: list[str]):
    """编码 texts，构建 FAISS IndexFlatIP，返回 (index, chunks)。"""
    # ✏️ TODO: 加载 SentenceTransformer('all-mpnet-base-v2')
    # ✏️ TODO: model.encode(texts) → float32 numpy array
    # ✏️ TODO: faiss.normalize_L2(vectors)
    # ✏️ TODO: index = faiss.IndexFlatIP(dim); index.add(vectors)
    # ✏️ TODO: return index, texts
    raise NotImplementedError

In [ ]:
# 检查 build_rag_index
docs = [
    "Chord progressions define the harmonic movement of a piece.",
    "MFCCs capture the timbral and spectral characteristics of audio.",
    "The Fourier transform decomposes a signal into frequency components.",
    "Attention mechanisms allow transformers to focus on relevant tokens.",
    "Mel filterbanks approximate human auditory perception of frequency.",
]
index, chunks = build_rag_index(docs)
assert index.ntotal == len(docs), f"期望 {len(docs)} 个向量，得到 {index.ntotal}"
assert len(chunks) == len(docs)
print(f'✅ build_rag_index：index.ntotal = {index.ntotal}，维度 = {index.d}')

## 5. ✏️ 实现 `retrieve(query, index, chunks, top_k=3)`

**签名**：`query: str, index: faiss.Index, chunks: list[str], top_k: int -> list[str]`

**推理路线**：
1. 加载（或复用缓存的）`SentenceTransformer` 模型，编码 `query` 为 `(1, 768)` 向量
2. `faiss.normalize_L2(query_vec)` 归一化（与索引向量同空间）
3. `index.search(query_vec, top_k)` 返回 `(distances, indices)`，形状均为 `(1, top_k)`
4. 用 `indices[0]` 取对应的 chunk 文本，过滤掉 index == -1（未找到时 FAISS 返回 -1）
5. 返回相关文本列表

**参考输入输出**：
```
query = "What is chord progression?"
→  返回包含和声（chord / harmonic）相关内容的 3 段
```

<details><summary>点击查看参考实现</summary>

```python
def retrieve(query: str, index, chunks: list[str], top_k: int = 3) -> list[str]:
    model = SentenceTransformer('all-mpnet-base-v2')
    q_vec = model.encode([query], convert_to_numpy=True,
                         normalize_embeddings=False).astype(np.float32)
    faiss.normalize_L2(q_vec)
    distances, indices = index.search(q_vec, top_k)
    return [chunks[i] for i in indices[0] if i != -1]
```

</details>

In [ ]:
def retrieve(query: str, index, chunks: list[str], top_k: int = 3) -> list[str]:
    """编码 query，在 index 中搜索，返回 top_k 最相关的 chunk。"""
    # ✏️ TODO: 加载 SentenceTransformer('all-mpnet-base-v2')
    # ✏️ TODO: 编码 query → (1, dim) float32
    # ✏️ TODO: faiss.normalize_L2(q_vec)
    # ✏️ TODO: index.search(q_vec, top_k) → distances, indices
    # ✏️ TODO: return [chunks[i] for i in indices[0] if i != -1]
    raise NotImplementedError

In [ ]:
# 检查 retrieve
results = retrieve("What is chord progression?", index, chunks, top_k=2)
assert isinstance(results, list), "应返回 list"
assert len(results) <= 2
print('✅ retrieve 返回结果：')
for i, r in enumerate(results, 1):
    print(f'  {i}. {r}')

## 6. 参数实验：索50段音乐理论文档，评估检索质量

**参数**：
- `top_k = 1, 3, 5`：观察召回率与精确率的权衡
- `chunk_size = 100 vs 512` words：小块精准但信息少，大块信息全但噪声多
- 查询多样性：专业术语（"MFCC"）vs 口语描述（"how does audio capture timbre"）

**预期现象**：
- `top_k` 越大，相关文档被找到的概率越高，但无关结果也更多
- 专业术语查询的精确率通常高于口语描述
- 向量归一化后，余弦相似度在 `[0, 1]` 范围，相关文档得分通常 > 0.6

In [ ]:
# 构建50段音乐/音频AI领域的模拟文档
import random
random.seed(42)

domain_docs = [
    "Chord progressions define the harmonic movement of a piece of music.",
    "MFCCs capture timbral and spectral characteristics of audio signals.",
    "The Fourier transform decomposes a time-domain signal into frequencies.",
    "Attention mechanisms in transformers weigh the relevance of each token.",
    "Mel filterbanks approximate the nonlinear human perception of pitch.",
    "Beat tracking algorithms detect the rhythmic pulse in audio streams.",
    "Spectrograms visualize how frequency content changes over time.",
    "Pitch class profiles, or chroma features, represent harmonic content.",
    "Neural audio codecs compress waveforms into discrete token sequences.",
    "Self-supervised learning pre-trains models on unlabeled audio data.",
] * 5  # 50 段

big_index, big_chunks = build_rag_index(domain_docs)
print(f'索引大小: {big_index.ntotal} 个向量\n')

queries = [
    "how do I analyze harmony in music?",
    "feature extraction from audio",
    "transformer model for audio",
]

for top_k in [1, 3]:
    print(f'=== top_k = {top_k} ===')
    for q in queries:
        hits = retrieve(q, big_index, big_chunks, top_k=top_k)
        print(f'  Q: "{q}"')
        for h in hits:
            print(f'    → {h[:60]}...')
    print()

## 本课小结

本课实现了 `build_rag_index`（编码 + L2归一化 + `faiss.IndexFlatIP.add`）和 `retrieve`（编码 query + `index.search` → top-k chunk），两者共同构成 `aurora.rag` 的检索核心。`build_rag_index` 输出一个 FAISS 索引和对应的文本块列表，`retrieve` 输出按相关性排序的文本段，直接可拼入 LLM prompt。下一节（M5-L5）将把这两个函数接入 Claude API，完成完整的 Podcast RAG Agent：用户提问 → 检索音频笔记 → LLM 生成带引用的回答。